# Google Colab Setup Guide

Run the cells in this notebook **once** to prepare your Colab environment before opening any other notebook in the project.

## What this notebook does
1. Mounts your Google Drive
2. Clones (or updates) the Factor-Research repository onto Drive
3. Installs all required Python packages
4. Verifies the data directory structure

## Data you need to supply
The notebooks expect the following raw data folders inside `data/raw/` of the repository:

```
data/
└── raw/
    ├── datamin2/          # Minute-level OHLCV CSVs (source 1)
    ├── datamin3/          # Minute-level OHLCV CSVs (source 2)
    └── datamin4/
        └── data_barra/   # Barra factor CSVs
```

Upload your data to the matching folders on Google Drive **before** running the pipeline notebooks.  
Each CSV must have columns: `datetime, order_book_id, open, high, low, close, volume, money`.

## Notebook execution order
1. `Data Preparation.ipynb`
2. `Alpha_Factor_Generation.ipynb`
3. `Alpha_Factor_Generation_2.ipynb`
4. `Alpha_Factor_Selection.ipynb`
5. `ML_Model.ipynb`
6. `Factor_backtest.ipynb`

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
# ── Step 2: Clone or update the repository ──────────────────────────────────
import os

REPO_PATH = '/content/drive/MyDrive/Factor-Research'
GITHUB_URL = 'https://github.com/mbrennan5/Factor-Research.git'

if not os.path.exists(REPO_PATH):
    print('Cloning repository — this may take a minute...')
    ret = os.system(f'git clone {GITHUB_URL} "{REPO_PATH}"')
    print('Clone finished.' if ret == 0 else f'Clone exited with code {ret}.')
else:
    print(f'Repository already exists at {REPO_PATH}')
    print('Pulling latest changes...')
    os.system(f'git -C "{REPO_PATH}" pull')

In [ ]:
# ── Step 3: Install required packages ───────────────────────────────────────
# pandas, numpy, scipy, scikit-learn, matplotlib, seaborn, torch are
# pre-installed in Colab.  Install the remaining ones:
print('Installing packages (this takes ~1-2 minutes)...')
!pip install -q lightgbm xgboost optuna plotly tqdm yfinance
print('All packages installed.')

In [ ]:
# ── Step 4: Verify directory structure ──────────────────────────────────────
import os

REPO_PATH = '/content/drive/MyDrive/Factor-Research'

required_dirs = [
    'data/raw/datamin2',
    'data/raw/datamin3',
    'data/raw/datamin4/data_barra',
    'data/processed',
    'data/factors/obtained_features',
    'notebooks',
]

print('Directory check:')
all_ok = True
for d in required_dirs:
    full = os.path.join(REPO_PATH, d)
    exists = os.path.isdir(full)
    if not exists:
        os.makedirs(full, exist_ok=True)
        status = '  CREATED'
    else:
        n_files = len([f for f in os.listdir(full) if not f.startswith('.')])
        status = f'  OK  ({n_files} files)'
        if 'raw/' in d and n_files == 0:
            status += '  ← upload your CSV data here'
            all_ok = False
    print(f'  {d:45s} {status}')

print()
if all_ok:
    print('Setup complete!  You can now run the pipeline notebooks in order.')
else:
    print('Upload your raw data CSV files to the folders marked above, then run the pipeline notebooks.')

In [ ]:
# ── Step 5: Quick sanity check ───────────────────────────────────────────────
# Verifies that the key packages import correctly.
import pandas as pd, numpy as np, scipy, sklearn, torch, lightgbm, xgboost, optuna
print('Package versions:')
print(f'  pandas      {pd.__version__}')
print(f'  numpy       {np.__version__}')
print(f'  torch       {torch.__version__}')
print(f'  lightgbm    {lightgbm.__version__}')
print(f'  xgboost     {xgboost.__version__}')
print(f'  optuna      {optuna.__version__}')
print(f'  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('  Tip: enable a GPU runtime (Runtime > Change runtime type > T4 GPU) for faster model training.')